# Template 04c: Feature Encoding

**Inputs:**
- data/04_train.parquet
- data/04_test.parquet
- config_generated/master_feature_encoding.csv

**Outputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- models/04c_encoders.pkl (for holdout)
- results/04c_encoding_summary.csv

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
from feature_encoder import apply_master_encoding, save_encoders

print("########################################")
print("# STAGE 04c: FEATURE ENCODING")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 04c: FEATURE ENCODING
########################################


In [4]:
config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
print(f'Output: {output_base}')

Output: output/car_coll/v1


In [5]:
# Load train/test splits
train_file = f'{output_base}/data/04_train.parquet'
test_file = f'{output_base}/data/04_test.parquet'

print(f'\n* Loading data...')
train = pd.read_parquet(train_file)
test = pd.read_parquet(test_file)

print(f'  Train: {train.shape}')
print(f'  Test: {test.shape}')


* Loading data...


  Train: (3757142, 122)
  Test: (3750266, 122)


In [6]:
# Apply master encoding
print(f'\n* Applying master feature encoding...')

train_encoded, test_encoded, encoders, encoding_summary = apply_master_encoding(
    train, test, config_path,
    min_frequency=0.01,  # 1% threshold
    min_count=50         # OR 50 observations
)


* Applying master feature encoding...
✓ Loaded master encoding: 406 columns



✓ Encoding complete:
  Train shape: (3757142, 197)
  Test shape: (3750266, 197)


In [7]:
# Display encoding summary
print(f'\n* Encoding Summary:')
print(f'\nEncoding types used:')
print(encoding_summary['encoding_type'].value_counts())

print(f'\nCategories with __OTHER__:')
print(encoding_summary[encoding_summary['has_other']== True][['original_column', 'n_categories']].head(10))

print(f'\nCategories with __MISSING__:')
print(encoding_summary[encoding_summary['has_missing'] == True][['original_column', 'n_categories']].head(10))


* Encoding Summary:

Encoding types used:
encoding_type
ordinal_0_5                30
one_hot                    30
binary                     20
remap_2to4_then_ordinal     5
numeric                     4
skip                        3
DROP                        2
Name: count, dtype: int64

Categories with __OTHER__:
                             original_column  n_categories
9                              NumMinAcc_raw             6
10                           NumMajinAcc_raw             6
11                            NumSpdViol_raw             6
12                            NumMinViol_raw            11
13                            NumMajViol_raw             7
25  vc_active_collision_avoidance_system_raw             4
26          vc_active_driving_assistance_raw             4
27          vc_active_parking_assistance_raw             4
28            vc_adaptive_cruise_control_raw             4
30  vc_audible_forward_collision_warning_raw             4

Categories with __MISSING__:


In [8]:
# Save encoded data
train_output = f'{output_base}/data/04c_train_encoded.parquet'
test_output = f'{output_base}/data/04c_test_encoded.parquet'

train_encoded.to_parquet(train_output, index=False)
test_encoded.to_parquet(test_output, index=False)

print(f'\n* Saved:')
print(f'  {train_output}')
print(f'  {test_output}')


* Saved:
  output/car_coll/v1/data/04c_train_encoded.parquet
  output/car_coll/v1/data/04c_test_encoded.parquet


In [9]:
# Save encoders for holdout
save_encoders(encoders, encoding_summary, output_base)

✓ Saved encoders: output/car_coll/v1/models/04c_encoders.pkl
✓ Saved encoding summary: output/car_coll/v1/results/04c_encoding_summary.csv
✓ Saved encoding map: output/car_coll/v1/models/04c_encoding_map.json


In [10]:
print("\n########################################")
print("# STAGE 04c: COMPLETE")
print("########################################")


########################################
# STAGE 04c: COMPLETE
########################################
